# Laboratorio 5 – Naive Bayes

**Continuación directa del Lab 4** — se reutilizan el mismo dataset preprocesado,
el mismo `train_test_split(random_state=42)` y la misma variable categórica `price_category`.


## Configuración
Reutilizar todo el pipeline del Lab 4

In [1]:
import pyreadr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (mean_squared_error, mean_absolute_error, r2_score,
                              accuracy_score, classification_report,
                              confusion_matrix, ConfusionMatrixDisplay)
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.cluster import KMeans

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110
print("Librerías cargadas.")


Librerías cargadas.


In [2]:
# PASO 1: Cargar datos
result = pyreadr.read_r('listings.Rdata')
df_raw = result[list(result.keys())[0]].copy()

# PASO 2: Limpiar precio
df = df_raw.copy()
if df['price'].dtype == object:
    df['price'] = (df['price'].str.replace(r'[\$,]', '', regex=True)
                   .str.strip().replace('', np.nan).astype(float))
q_high = df['price'].quantile(0.99)
df = df[(df['price'] > 0) & (df['price'] <= q_high)].copy()

# PASO 3: Eliminar columnas irrelevantes
cols_to_drop = ['id','listing_url','scrape_id','last_scraped','source','name',
    'description','neighborhood_overview','picture_url','host_url',
    'host_thumbnail_url','host_picture_url','host_about','host_verifications',
    'amenities','calendar_updated','calendar_last_scraped','license',
    'bathrooms_text','minimum_minimum_nights','maximum_minimum_nights',
    'minimum_maximum_nights','maximum_maximum_nights',
    'minimum_nights_avg_ntm','maximum_nights_avg_ntm']
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])
null_pct = df.isnull().mean()
df = df.drop(columns=null_pct[null_pct > 0.60].index.tolist())

# PASO 4: Ingeniería de fechas
if 'host_since' in df.columns:
    df['host_since'] = pd.to_datetime(df['host_since'], errors='coerce')
    df['host_years'] = ((pd.Timestamp('2024-01-01') - df['host_since']).dt.days / 365).round(1)
    df = df.drop(columns=['host_since'])
df = df.drop(columns=[c for c in ['first_review','last_review'] if c in df.columns], errors='ignore')

# PASO 5: Booleanos y tasas
for col in ['host_is_superhost','host_has_profile_pic','host_identity_verified',
            'has_availability','instant_bookable']:
    if col in df.columns:
        df[col] = df[col].map({'t':1,'f':0,True:1,False:0})
for col in ['host_response_rate','host_acceptance_rate']:
    if col in df.columns:
        df[col] = df[col].str.replace('%','',regex=False).str.strip().astype(float, errors='ignore')

# PASO 6: Selección y OHE
TARGET = 'price'
num_features = [c for c in df.select_dtypes(include='number').columns if c != TARGET]
cat_features = [c for c in ['room_type','property_type','neighbourhood_cleansed',
                              'host_response_time'] if c in df.columns]
for col in cat_features:
    freq = df[col].value_counts(normalize=True)
    df[col] = df[col].replace(freq[freq < 0.01].index, 'Otro')
df_encoded = pd.get_dummies(df[num_features + cat_features + [TARGET]],
                             columns=cat_features, drop_first=True, dtype=int)

# PASO 7: Imputar nulos
for col in df_encoded.columns:
    if df_encoded[col].isnull().any():
        df_encoded[col].fillna(df_encoded[col].median(), inplace=True)

# PASO 8: Variable categórica (igual que Lab 4)
p33 = df_encoded[TARGET].quantile(0.33)
p67 = df_encoded[TARGET].quantile(0.67)
def categorize_price(p):
    if p <= p33: return 'Económico'
    elif p <= p67: return 'Intermedio'
    else: return 'Caro'
df_encoded['price_category'] = df_encoded[TARGET].apply(categorize_price)

# PASO 9: Split IDÉNTICO al Lab 4 (random_state=42, test_size=0.20)
feature_cols = [c for c in df_encoded.columns if c not in [TARGET, 'cluster', 'price_category']]
X = df_encoded[feature_cols]
y = df_encoded[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

# PASO 10: Split clasificación (estratificado, igual que Lab 4)
le = LabelEncoder()
y_clf_enc = le.fit_transform(df_encoded['price_category'])
X_clf = df_encoded[feature_cols]
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_clf, y_clf_enc, test_size=0.20, random_state=42, stratify=y_clf_enc)

print(f"Dataset: {df_encoded.shape[0]:,} filas × {len(feature_cols)} features")
print(f"Train regresión:      {len(X_train):,}   Test: {len(X_test):,}")
print(f"Train clasificación:  {len(X_train_c):,}   Test: {len(X_test_c):,}")
print(f"Categorías: P33=${p33:.0f}  P67=${p67:.0f}")
print(f"Clases: {dict(zip(le.classes_, le.transform(le.classes_)))}")
print("Pipeline Lab 4.")


Dataset: 75,531 filas × 73 features
Train regresión:      60,424   Test: 15,107
Train clasificación:  60,424   Test: 15,107
Categorías: P33=$140  P67=$267
Clases: {'Caro': 0, 'Económico': 1, 'Intermedio': 2}
Pipeline Lab 4.


In [3]:
# Re-entrenar modelos del Lab 4 para tener sus predicciones disponibles
# Árbol de regresión (mejor: depth=12)
t_start = time.time()
best_tree_reg = DecisionTreeRegressor(max_depth=12, min_samples_split=50, random_state=42)
best_tree_reg.fit(X_train, y_train)
yp_tree_reg = best_tree_reg.predict(X_test)
t_tree_reg = time.time() - t_start

# Random Forest regresión
t_start = time.time()
rf_reg = RandomForestRegressor(n_estimators=200, max_depth=15, min_samples_split=10,
                                min_samples_leaf=5, max_features='sqrt', random_state=42, n_jobs=-1)
rf_reg.fit(X_train, y_train)
yp_rf_reg = rf_reg.predict(X_test)
t_rf_reg = time.time() - t_start

# Ridge Regresión
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)
y_train_log = np.log1p(y_train)
t_start = time.time()
ridge_cv = RidgeCV(alphas=np.logspace(-3,3,50), cv=5)
ridge_cv.fit(X_train_sc, y_train_log)
yp_ridge = np.expm1(ridge_cv.predict(X_test_sc))
t_ridge = time.time() - t_start

# Árbol de clasificación (mejor: depth=15)
t_start = time.time()
best_tree_clf = DecisionTreeClassifier(max_depth=15, random_state=42)
best_tree_clf.fit(X_train_c, y_train_c)
yp_tree_clf = best_tree_clf.predict(X_test_c)
t_tree_clf = time.time() - t_start

# Random Forest clasificación
t_start = time.time()
rf_clf = RandomForestClassifier(n_estimators=200, max_depth=15, min_samples_split=10,
                                 min_samples_leaf=5, max_features='sqrt', random_state=42, n_jobs=-1)
rf_clf.fit(X_train_c, y_train_c)
yp_rf_clf = rf_clf.predict(X_test_c)
t_rf_clf = time.time() - t_start

print("Modelos del Lab anterior.")
print(f"  Árbol regresión:   {t_tree_reg:.2f}s")
print(f"  Random Forest reg: {t_rf_reg:.2f}s")
print(f"  Ridge:             {t_ridge:.2f}s")
print(f"  Árbol clasif.:     {t_tree_clf:.2f}s")
print(f"  Random Forest clf: {t_rf_clf:.2f}s")


Modelos del Lab anterior.
  Árbol regresión:   0.90s
  Random Forest reg: 6.33s
  Ridge:             17.07s
  Árbol clasif.:     1.18s
  Random Forest clf: 7.15s
